# Baseline dataset (no text)

In [ ]:
# ============================================================
# SCRIPT 1: BUILD FINAL STRUCTURED DATASETS
# ============================================================

# ------------------------------------------------------------
# 0. IMPORTS AND DRIVE
# ------------------------------------------------------------
from google.colab import drive
import pandas as pd
import numpy as np

drive.mount('/content/gdrive', force_remount=True)

# ------------------------------------------------------------
# 1. LOAD RAW DATA
# ------------------------------------------------------------
path_e0 = '/content/gdrive/MyDrive/Speciale/E0.csv'
path_e1 = '/content/gdrive/MyDrive/Speciale/E1.csv'
path_e2 = '/content/gdrive/MyDrive/Speciale/E2.csv'

e0 = pd.read_csv(path_e0)
e1 = pd.read_csv(path_e1)
e2 = pd.read_csv(path_e2)

print("Raw shapes:")
print("E0:", e0.shape)
print("E1:", e1.shape)
print("E2:", e2.shape)

# Add season label
e0["season"] = "2022_2023"
e1["season"] = "2023_2024"
e2["season"] = "2024_2025"

# Combine
matches_full = pd.concat([e0, e1, e2], ignore_index=True)

print("\nCombined shape:", matches_full.shape)

# ------------------------------------------------------------
# 2. KEEP ONLY REQUIRED RAW COLUMNS
# ------------------------------------------------------------
raw_keep_cols = [
    "Date", "Time", "HomeTeam", "AwayTeam",
    "FTHG", "FTAG", "FTR",
    "AvgH", "AvgD", "AvgA",
    "season"
]

matches_full = matches_full[raw_keep_cols].copy()

print("\nColumns kept:")
print(matches_full.columns.tolist())

# ------------------------------------------------------------
# 3. CLEAN DATETIME
# ------------------------------------------------------------
# Some rows may have missing Time; fill with a safe default
matches_full["Time"] = matches_full["Time"].fillna("15:00")

matches_full["MatchDatetime"] = pd.to_datetime(
    matches_full["Date"].astype(str) + " " + matches_full["Time"].astype(str),
    dayfirst=True,
    errors="coerce"
)

# Drop rows where datetime failed
matches_full = matches_full.dropna(subset=["MatchDatetime"]).reset_index(drop=True)

# Sort chronologically
matches_full = matches_full.sort_values("MatchDatetime").reset_index(drop=True)

print("\nDate range:")
print(matches_full["MatchDatetime"].min(), "to", matches_full["MatchDatetime"].max())

# ------------------------------------------------------------
# 4. BUILD FEATURES
# ------------------------------------------------------------
# Recent window
n_recent = 5

# Empty collectors
home_attack = []
home_defence = []
away_attack = []
away_defence = []

home_ppg = []
away_ppg = []

home_gdpg = []
away_gdpg = []

home_form_ppg = []
away_form_ppg = []

# Team history store
team_stats = {}

for _, row in matches_full.iterrows():
    home = row["HomeTeam"]
    away = row["AwayTeam"]

    # Initialise team records
    for team in [home, away]:
        if team not in team_stats:
            team_stats[team] = {
                "goals_scored": [],
                "goals_conceded": [],
                "points": []
            }

    # Helper functions
    def get_avg(team, key, fallback):
        values = team_stats[team][key]
        return np.mean(values) if len(values) > 0 else fallback

    def get_recent_avg(team, key, n, fallback):
        values = team_stats[team][key]
        if len(values) == 0:
            return fallback
        return np.mean(values[-n:])

    # -------------------------
    # PRE-MATCH LONG-TERM FEATURES
    # -------------------------
    h_att = get_avg(home, "goals_scored", fallback=1.0)
    h_def = get_avg(home, "goals_conceded", fallback=1.0)
    h_ppg = get_avg(home, "points", fallback=1.0)
    h_gdpg = h_att - h_def

    a_att = get_avg(away, "goals_scored", fallback=1.0)
    a_def = get_avg(away, "goals_conceded", fallback=1.0)
    a_ppg = get_avg(away, "points", fallback=1.0)
    a_gdpg = a_att - a_def

    # -------------------------
    # PRE-MATCH RECENT FORM FEATURES
    # -------------------------
    h_form_ppg = get_recent_avg(home, "points", n=n_recent, fallback=1.0)
    a_form_ppg = get_recent_avg(away, "points", n=n_recent, fallback=1.0)

    # Store pre-match features
    home_attack.append(h_att)
    home_defence.append(h_def)
    away_attack.append(a_att)
    away_defence.append(a_def)

    home_ppg.append(h_ppg)
    away_ppg.append(a_ppg)

    home_gdpg.append(h_gdpg)
    away_gdpg.append(a_gdpg)

    home_form_ppg.append(h_form_ppg)
    away_form_ppg.append(a_form_ppg)

    # -------------------------
    # UPDATE AFTER MATCH
    # -------------------------
    if row["FTR"] == "H":
        home_points = 3
        away_points = 0
    elif row["FTR"] == "D":
        home_points = 1
        away_points = 1
    else:
        home_points = 0
        away_points = 3

    # Update home history
    team_stats[home]["goals_scored"].append(row["FTHG"])
    team_stats[home]["goals_conceded"].append(row["FTAG"])
    team_stats[home]["points"].append(home_points)

    # Update away history
    team_stats[away]["goals_scored"].append(row["FTAG"])
    team_stats[away]["goals_conceded"].append(row["FTHG"])
    team_stats[away]["points"].append(away_points)

# Add engineered features
matches_full["Home_attack"] = home_attack
matches_full["Home_defence"] = home_defence
matches_full["Away_attack"] = away_attack
matches_full["Away_defence"] = away_defence

matches_full["Home_PPG"] = home_ppg
matches_full["Away_PPG"] = away_ppg

matches_full["Home_GDpg"] = home_gdpg
matches_full["Away_GDpg"] = away_gdpg

matches_full["Home_form_PPG"] = home_form_ppg
matches_full["Away_form_PPG"] = away_form_ppg

# ------------------------------------------------------------
# 5. DIFFERENCE / INTERACTION / CLOSENESS FEATURES
# ------------------------------------------------------------
matches_full["Attack_diff"] = matches_full["Home_attack"] - matches_full["Away_attack"]
matches_full["Defence_diff"] = matches_full["Home_defence"] - matches_full["Away_defence"]
matches_full["PPG_diff"] = matches_full["Home_PPG"] - matches_full["Away_PPG"]
matches_full["GDpg_diff"] = matches_full["Home_GDpg"] - matches_full["Away_GDpg"]

matches_full["Form_PPG_diff"] = matches_full["Home_form_PPG"] - matches_full["Away_form_PPG"]

matches_full["Interaction_home"] = matches_full["Home_attack"] - matches_full["Away_defence"]
matches_full["Interaction_away"] = matches_full["Away_attack"] - matches_full["Home_defence"]

matches_full["Strength_diff_abs"] = matches_full["PPG_diff"].abs()

# ------------------------------------------------------------
# 6. FINAL DATASET WITHOUT ODDS
# ------------------------------------------------------------
final_cols_no_odds = [
    "MatchDatetime",
    "season",
    "HomeTeam",
    "AwayTeam",
    "FTR",

    "Home_attack",
    "Home_defence",
    "Away_attack",
    "Away_defence",

    "Home_PPG",
    "Away_PPG",
    "Home_GDpg",
    "Away_GDpg",

    "Home_form_PPG",
    "Away_form_PPG",

    "Attack_diff",
    "Defence_diff",
    "PPG_diff",
    "GDpg_diff",
    "Form_PPG_diff",

    "Interaction_home",
    "Interaction_away",
    "Strength_diff_abs"
]

matches_final_no_odds = matches_full[final_cols_no_odds].copy()

# ------------------------------------------------------------
# 7. FINAL DATASET WITH ODDS
# ------------------------------------------------------------
final_cols_with_odds = final_cols_no_odds + [
    "AvgH", "AvgD", "AvgA"
]

matches_final_with_odds = matches_full[final_cols_with_odds].copy()

# ------------------------------------------------------------
# 8. QUICK CHECKS
# ------------------------------------------------------------
print("\nNo-odds dataset shape:", matches_final_no_odds.shape)
print(matches_final_no_odds.head())

print("\nWith-odds dataset shape:", matches_final_with_odds.shape)
print(matches_final_with_odds.head())

print("\nMissing values in no-odds dataset:")
print(matches_final_no_odds.isna().sum().sort_values(ascending=False).head(20))

print("\nMissing values in with-odds dataset:")
print(matches_final_with_odds.isna().sum().sort_values(ascending=False).head(20))

print("\nTarget distribution:")
print(matches_final_no_odds["FTR"].value_counts(normalize=True).sort_index())

# ------------------------------------------------------------
# 9. EXPORT
# ------------------------------------------------------------
path_no_odds = "/content/gdrive/MyDrive/Speciale/matches_final_no_odds.csv"
path_with_odds = "/content/gdrive/MyDrive/Speciale/matches_final_with_odds.csv"
path_no_odds1 = "/content/gdrive/MyDrive/Speciale/matches_final_no_odds.xlsx"
path_with_odds1 = "/content/gdrive/MyDrive/Speciale/matches_final_with_odds.xlsx"

matches_final_no_odds.to_csv(path_no_odds, index=False)
matches_final_with_odds.to_csv(path_with_odds, index=False)
matches_final_no_odds.to_excel(path_no_odds1, index=False)
matches_final_with_odds.to_excel(path_with_odds1, index=False)

print("\nSaved files:")
print(path_no_odds)
print(path_with_odds)

# Final datasets (text added)

In [ ]:
# ------------------------------------------------------------
# 0. IMPORTS AND DRIVE
# ------------------------------------------------------------
from google.colab import drive
import pandas as pd
import numpy as np

drive.mount('/content/gdrive', force_remount=True)


Mounted at /content/gdrive


In [ ]:
# File paths
path_with_odds = "/content/gdrive/MyDrive/Speciale/matches_final_with_odds(text).xlsx"
path_no_odds   = "/content/gdrive/MyDrive/Speciale/matches_final_no_odds(text).xlsx"

# Load and convert "-" to NaN immediately
df_with_odds = pd.read_excel(path_with_odds, na_values=["-"])
df_no_odds   = pd.read_excel(path_no_odds, na_values=["-"])

In [ ]:
print(df_with_odds)

           MatchDatetime     season        HomeTeam        AwayTeam FTR  \
0    2022-08-05 20:00:00  2022_2023  Crystal Palace         Arsenal   A   
1    2022-08-06 12:30:00  2022_2023          Fulham       Liverpool   D   
2    2022-08-06 15:00:00  2022_2023     Bournemouth     Aston Villa   H   
3    2022-08-06 15:00:00  2022_2023           Leeds          Wolves   H   
4    2022-08-06 15:00:00  2022_2023       Newcastle   Nott'm Forest   H   
...                  ...        ...             ...             ...  ..   
1135 2025-05-25 16:00:00  2024_2025         Ipswich        West Ham   A   
1136 2025-05-25 16:00:00  2024_2025          Fulham        Man City   A   
1137 2025-05-25 16:00:00  2024_2025     Bournemouth       Leicester   H   
1138 2025-05-25 16:00:00  2024_2025       Liverpool  Crystal Palace   D   
1139 2025-05-25 16:00:00  2024_2025          Wolves       Brentford   D   

      Home_attack  Home_defence  Away_attack  Away_defence  Home_PPG  ...  \
0        1.000000     

In [ ]:
# No odds
df_no_odds_clean = df_no_odds[
    df_no_odds["factual_text"].notna() &
    df_no_odds["opinion_text"].notna()
].copy()

In [ ]:
# With odds
df_with_odds_clean = df_with_odds[
    df_with_odds["factual_text"].notna() &
    df_with_odds["opinion_text"].notna() &
    df_with_odds["AvgH"].notna() &
    df_with_odds["AvgD"].notna() &
    df_with_odds["AvgA"].notna()
].copy()

In [ ]:
# ============================================================
# FULL FACTUAL FEATURE ENGINEERING
# ============================================================

import re
import pandas as pd

# ------------------------------------------------------------
# Helper: clean text
# ------------------------------------------------------------
def clean_text(x):
    return (
        str(x)
        .replace("\xa0", " ")
        .replace("\r", "\n")
        .replace("\n\n", "\n")
        .strip()
    )


# ------------------------------------------------------------
# Helper: split into Home / Away using team names
# ------------------------------------------------------------
def split_home_away(row):
    text = clean_text(row["factual_text"])
    home_team = str(row["HomeTeam"]).strip()
    away_team = str(row["AwayTeam"]).strip()

    try:
        pattern = rf"{re.escape(home_team)}:(.*?){re.escape(away_team)}:(.*)"
        match = re.search(pattern, text, re.IGNORECASE | re.DOTALL)

        if match:
            home_text = match.group(1).strip()
            away_text = match.group(2).strip()
            return home_text, away_text

        return "", ""

    except Exception:
        return "", ""


# ------------------------------------------------------------
# Extract named section
# ------------------------------------------------------------
def extract_section(text, section_name):
    text = clean_text(text)

    pattern = (
        rf"{section_name}\s+"
        rf"(.*?)"
        rf"(?=Injured|Doubtful|Suspended|Unavailable|Form|Leading scorer|Subs from|$)"
    )

    match = re.search(pattern, text, re.IGNORECASE | re.DOTALL)

    if match:
        return match.group(1).strip(" .,\n\t")

    return ""


# ------------------------------------------------------------
# Count players in a section
# ------------------------------------------------------------
def count_players(section):
    section = clean_text(section)

    if not section:
        return 0

    section_lower = section.lower().strip(" .,\n\t")

    if section_lower in ["none", "no one", "n/a", "na", "-"]:
        return 0

    players = [p.strip() for p in section.split(",") if p.strip()]

    if len(players) == 1 and players[0].lower().strip(" .,\n\t") in ["none", "no one", "n/a", "na", "-"]:
        return 0

    return len(players)


# ------------------------------------------------------------
# Normalise player name/text for matching
# ------------------------------------------------------------
def normalise_name(x):
    x = str(x).lower()
    x = re.sub(r"[^a-zA-ZÀ-ÿ\s'-]", " ", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x


# ------------------------------------------------------------
# Check whether top scorer is listed as missing
# ------------------------------------------------------------
def is_top_scorer_missing(scorer_name, missing_text):
    scorer_name = normalise_name(scorer_name)
    missing_text = normalise_name(missing_text)

    if not scorer_name or not missing_text:
        return 0

    scorer_parts = scorer_name.split()

    if not scorer_parts:
        return 0

    surname = scorer_parts[-1]

    full_name_pattern = rf"\b{re.escape(scorer_name)}\b"
    surname_pattern = rf"\b{re.escape(surname)}\b"

    full_name_match = re.search(full_name_pattern, missing_text, re.IGNORECASE) is not None
    surname_match = re.search(surname_pattern, missing_text, re.IGNORECASE) is not None

    return int(full_name_match or surname_match)


# ------------------------------------------------------------
# Extract features from one team block
# ------------------------------------------------------------
def extract_team_features(text):

    text = clean_text(text)

    injured_text = extract_section(text, "Injured")
    doubtful_text = extract_section(text, "Doubtful")
    suspended_text = extract_section(text, "Suspended")
    unavailable_text = extract_section(text, "Unavailable")

    form = re.search(r"Form\s+([WDL]+)", text, re.IGNORECASE)

    scorer = re.search(
        r"Leading scorer\s+(.+?)\s+(\d+)",
        text,
        re.IGNORECASE
    )

    features = {}

    features["n_injured"] = count_players(injured_text)
    features["n_doubtful"] = count_players(doubtful_text)
    features["n_suspended"] = count_players(suspended_text)
    features["n_unavailable"] = count_players(unavailable_text)

    if form:
        form_str = form.group(1).upper()
        features["form_points"] = sum(
            3 if x == "W" else 1 if x == "D" else 0
            for x in form_str
        )
    else:
        features["form_points"] = 0

    if scorer:
        scorer_name = scorer.group(1).strip()
        scorer_goals = int(scorer.group(2))

        all_missing_text = " ".join([
            injured_text,
            doubtful_text,
            suspended_text,
            unavailable_text
        ])

        features["top_scorer_goals"] = scorer_goals
        features["top_scorer_missing"] = is_top_scorer_missing(
            scorer_name,
            all_missing_text
        )

    else:
        features["top_scorer_goals"] = 0
        features["top_scorer_missing"] = 0

    return features


# ------------------------------------------------------------
# Apply factual feature engineering to dataframe
# ------------------------------------------------------------
def add_factual_features(df):

    df = df.copy()

    home_features = []
    away_features = []

    for _, row in df.iterrows():
        home_text, away_text = split_home_away(row)

        home_feat = extract_team_features(home_text)
        away_feat = extract_team_features(away_text)

        home_features.append(home_feat)
        away_features.append(away_feat)

    home_df = pd.DataFrame(home_features).add_prefix("home_")
    away_df = pd.DataFrame(away_features).add_prefix("away_")

    df = pd.concat(
        [df.reset_index(drop=True), home_df, away_df],
        axis=1
    )

    df["diff_injured"] = df["home_n_injured"] - df["away_n_injured"]
    df["diff_doubtful"] = df["home_n_doubtful"] - df["away_n_doubtful"]
    df["diff_suspended"] = df["home_n_suspended"] - df["away_n_suspended"]
    df["diff_unavailable"] = df["home_n_unavailable"] - df["away_n_unavailable"]
    df["diff_form_points"] = df["home_form_points"] - df["away_form_points"]
    df["diff_top_scorer_goals"] = (
        df["home_top_scorer_goals"] - df["away_top_scorer_goals"]
    )
    df["diff_top_scorer_missing"] = (
        df["home_top_scorer_missing"] - df["away_top_scorer_missing"]
    )

    return df


# ------------------------------------------------------------
# APPLY
# ------------------------------------------------------------
df_no_odds_clean = add_factual_features(df_no_odds_clean)
df_with_odds_clean = add_factual_features(df_with_odds_clean)

In [ ]:
# ------------------------------------------------------------
# SAVE FINAL LOCKED DATASETS
# ------------------------------------------------------------

df_no_odds_clean.to_csv(
    "/content/gdrive/MyDrive/Speciale/final_no_odds_dataset.csv",
    index=False
)

df_with_odds_clean.to_csv(
    "/content/gdrive/MyDrive/Speciale/final_with_odds_dataset.csv",
    index=False
)